# CAES mini-project: system design and simulation

Compressed Air Energy Storage (CAES) stores electricity by compressing air into a reservoir. During discharge, the air is heated and expanded through a turbine to produce electricity. The project compares how compression heat management and reservoir design affect performance.

## System variants

The six variants combine three architectures with two reservoir types:

| Architecture | Constant-volume reservoir | Constant-pressure reservoir |
|---|---|---|
| Single-stage compression and expansion; compression heat rejected | 1a | 1b |
| Single-stage with thermal energy storage (TES) | 2a | 2b |
| Two-stage compression with intercooling and two-stage expansion with reheat; no TES | 3a | 3b |

A constant-volume reservoir operates between 16 bar and a minimum pressure of 8 bar, so its pressure falls during discharge. A constant-pressure reservoir remains at 16 bar while its volume varies. Variants 2a and 2b store compression heat in a hot-water TES and reuse it to preheat discharge air before combustion. Variants 3a and 3b use equal pressure-ratio stages; intercooling heat is rejected, and fuel reheats the air before each turbine stage.

## Common specification

- Working fluid: real-gas air (CoolProp).
- Ambient conditions: 15°C and 1.013 bar.
- Maximum storage pressure: 16 bar; minimum pressure for constant-volume storage: 8 bar.
- Target output: 1 MW at the turbine/generator shaft for 4 hours (4 MWh per cycle); charge duration is also 4 hours.
- Compressor isentropic efficiency: 0.85; turbine isentropic efficiency: 0.88.
- Mechanical/generator efficiency: 0.98.
- Turbine inlet temperature: 550°C. Combustion is represented as heat addition to reach this temperature, without modelling combustion chemistry.
- Cooler and TES return pinches: 10 K. Air is cooled towards ambient temperature plus the cooler pinch before storage.
- The reservoir is treated as a uniform 0D store. Stored air is assumed to return to ambient temperature before discharge and remain near ambient during discharge.

## Performance and design tasks

Use the same two-step workflow for the selected variant:

1. **Design:** size the required components, heat exchangers, and storage volume for the common output target.
2. **Simulation:** model a complete charge/discharge cycle and evaluate performance.

The common round-trip metric values fuel at its electrical equivalent:

\[
\eta = \frac{W_{\mathrm{turbine}}}{W_{\mathrm{compressor}} + \eta_f Q_{\mathrm{combustion}}},
\qquad \eta_f = 0.40.
\]

Compare round-trip efficiency, storage volume, heat-exchanger areas, maximum temperatures, and the electrical/fuel input split. Include energy (and optionally exergy) balances, identify major losses, and quantify at least one proposed performance improvement. The final comparison should show how thermal storage, staging, and reservoir type affect these results.

Here are some sources i used to return to later (soso)

Return to Gas Turbines EENG4 S8 for compressor eq and details (soso)

Paper on which sile will the pinch temp be occuring based on the Cc and Ch (LMTD) method. https://sci-hub.fr/10.1016/B978-008042839-0/50008-2 (soso)

Also return to EENG3 S6 Lec 4 for counter flow details and eq (soso)

In [203]:
# --- imports ---
from pathlib import Path
import sys

for root in (Path.cwd(), *Path.cwd().parents):
    for package_root in (root, root / "caldis-0.3.0"):
        if (package_root / "caldis" / "__init__.py").is_file():
            sys.path.insert(0, str(package_root))
            break
    else:
        continue
    break

import numpy as np
import caldis
from CoolProp.CoolProp import PropsSI
from caldis.core.system import System
from caldis.components import FluidBoundary
from caldis.components import Compressor
from caldis.fluids import CoolPropBackend
from caldis.components import SizingHX
from caldis.components import CombustionChamber
from caldis.components import Turbine
from caldis.components.vessels.tank import TankConstP
from caldis.solvers import dynamic
from caldis.solvers import solve, simulate


Initilize paramaters

Ambient Air --> Compressor --> Cooler --> Air Storage --> Combustor --> Turbine --> Exhuast

In [204]:
# --- system definition ---
#Inlet Compressor 
T0=15 +273.15 #k
P0=1.013e5
Eta_ISC=0.85

#Cooler (water)
Tpinch=10 #k


#Air Storage
Pmax=16e5
    #Volume is Variable probably use constraint to find volume based on mass flow rate and time of storage

#Combustion Chamber
Ttit=550 +273.15 #k
Eta_Fuel=0.4

#Turbine
Eta_IST=0.85
Eta_Mech=0.98



Paramters to assume

In [205]:
# A1: First we define a mass flow rate
Mdot_air=1 #kg/s
Mdot_water=1 #kg/s

# Air storage sizing
Mdot_discharge_start = 2.54  # kg/s
Mass = Mdot_discharge_start * 3600 * 4  # kg stored for 4 hours

rho_tank_initial = PropsSI(
    'D', 'T', T0, 'P', Pmax, 'Air'
)  # kg/m3

V_tank_initial = Mass / rho_tank_initial  # m3


# A2: We define a guess for the water outlet temperature
T_water_out_Guess = 100 + 273.15 #K This is an assumption cuz it must be less than 400 and more than T0+Tpinch

# A3: We define the backend for the fluids which is the film convection coefficients
# The heat exchanger is a counter flow heat exchanger with water on one side and air on the other side. 
# The heat transfer coefficients are based on the type of fluid and the flow conditions. Typical values are:
#   forced convection in gas (air): 50–250 W/m²/K
#   forced convection in liquid (water): 500–10000 W/m²/K
h_film_air   = 100.0    # W/m²/K  (air side)
h_film_water = 3000.0   # W/m²/K  (water side)

T_out_comp_guess = 400 + 273.15 #K This is an assumption cuz it must be less than Ttit and more than T0

Ratio_Area_cold_to_hot = 0.5 # This is an assumption, it can be changed to see the effect on the performance of the system


In [206]:
# --- Component setup ---
be_air   = CoolPropBackend("Air")
be_water = CoolPropBackend("Water")
be_fuel = CoolPropBackend("Methane")

h_tank = PropsSI('H', 'T', T0, 'P', Pmax, 'Air')
Source = FluidBoundary(name="Source", T=T0, mdot=0, backend=be_air)

Tank = TankConstP(name="Tank", backend=be_air, p=Pmax, h_init=h_tank, V_init=V_tank_initial, C_wall=10 , T_wall_init=T0, UA_wall=1000, Q_heater=0)

#Source = FluidBoundary(name="Source", p=Pmax, T=T0, h=None, mdot=Turbine.ou, backend=be_air)

Comb= CombustionChamber(name="Comb", backend_air=be_air, backend_fuel=be_fuel, LHV=50e6, eta_comb=1.0, dp_rel=0.04)
Fuel_in = FluidBoundary(name="Fuel_in",p=17e5,T=T0, backend=be_fuel)

Turbine = Turbine(name="Turbine", backend=be_air, eta_is=Eta_IST, eta_mech=Eta_Mech)

Exhaust= FluidBoundary(name="Exhaust", p=P0, T=None, h=None, mdot=None, backend=be_air)


In [ ]:
# --- System wiring & start points ---
sys_ = System("Comp_Parts")
sys_.add(Source,Tank, Turbine, Comb, Fuel_in,Exhaust)
sys_.connect(Source.port, Tank.inl)

sys_.connect(Tank.out, Comb.air_in)
sys_.connect(Fuel_in.port, Comb.fuel_in)

sys_.connect(Comb.gas_out, Turbine.inl)
sys_.connect(Turbine.out, Exhaust.port)

# --- Constraints ---
# A. Reservoir inlet is closed during discharge (no air entering)
#sys_.add_constraint(lambda: Tank.inl.h - h_tank,  "tank_inlet_enthalpy")
# B. Combustion chamber brings gas to 550°C (TIT)
sys_.add_constraint(
    lambda: be_air.T(Comb.gas_out.p, Comb.gas_out.h) - Ttit,
    "TIT_550C")
W_target=1e6
# C. Turbine delivers 1 MW net shaft power
sys_.add_constraint(
    lambda: Eta_Mech * Turbine.inl.mdot * (Turbine.inl.h - Turbine.out.h) - W_target,
    "Wdot_1MW"
)

                    
# --- Start Hints (mdot ~ 2.51 kg/s air, ~ 0.0286 kg/s fuel) ---
sys_.set_start_port(Tank.out,   p=Pmax,        T=T0,          mdot=-2.54)
# Fuel entering the combustor
sys_.set_start_port(Fuel_in.port, p=17e5, T=T0, mdot=0.0286)
sys_.set_start_port(Comb.air_in,   p=Pmax,        T=T0,          mdot=2.54)
sys_.set_start_port(Comb.gas_out,  p=Pmax*0.96, T=Ttit,        mdot=-2.54)

sys_.set_start_port(Turbine.inl,      p=Pmax*0.96, T=Ttit,        mdot=2.54)
sys_.set_start_port(Turbine.out,      p=P0,          T=250+273.15, mdot=-2.54)
sys_.set_start_port(Exhaust.port,  p=P0,          T=250+273.15, mdot=2.54)

# Initialize the model's starting guesses once
sys_.init_starts()

# Run a 4-hour discharge simulation
results = simulate( sys_, t_end=4 * 3600, dt=60, init="consistent",progress=True)

print(type(results))
print(results)




sol = solve(sys_)

NameError: name 'simulate' is not defined

In [ ]:
#sys_.structure()
Tank.structure()




TankConstP 'Tank'
  outputs ['Q', 'T_fluid', 'T_wall', 'V']
  ControlVolumeP 'fluid'
    var    M  [state]
    var    U  [state]
    var    V
    var    h
    port   inl  (Air)   -> .p .mdot .h
    port   out  (Air)   -> .p .mdot .h
    outputs ['T', 'V', 'rho', 'x']
  ThermalMass 'wall'
    var    T  [state]
    outputs ['Qdot_ext']


In [ ]:
Comb.describe()

Turbine.describe()

Exhaust.describe()

=== CombustionChamber('Comb') ===
  ports (p [bar], mdot [kg/s], T [°C]):
    air_in     p=16.000  mdot=+2.54  h=410.7kJ/kg  T=15.00
    fuel_in    p=1.000  mdot=+0.01  h=876.7kJ/kg  T=10.00
    gas_out    p=15.360  mdot=-2.54  h=974.4kJ/kg  T=550.00
  outputs:
    Qdot_fuel        = 500000
    Qdot_released    = 500000
    AFR              = 254
    excess_air       = 14.7674
=== Turbine('Turbine') ===
  ports (p [bar], mdot [kg/s], T [°C]):
    inl        p=15.360  mdot=+2.54  h=974.4kJ/kg  T=550.00
    out        p=1.013  mdot=-2.54  h=653.3kJ/kg  T=250.00
  outputs:
    Wdot             = 799381
    Wdot_is          = 1.13619e+06
    pressure_ratio   = 15.1629
    h_out_s          = 517981
    T_out_s          = 390.852
=== FluidBoundary('Exhaust') ===
  ports (p [bar], mdot [kg/s], T [°C]):
    port       p=1.013  mdot=+2.54  h=653.3kJ/kg  T=250.00


{'variables': {},
 'ports': {'port': {'p': 101300.0, 'mdot': 2.54, 'h': 653288.7156722012}},
 'outputs': {}}